# Load Data from Bronze and Silver Layers

This step loads the data from the Bronze and Silver tables into separate Spark DataFrames.

The Bronze data will be used as a reference for data validation, including record count consistency checks.

The Silver data will be used for data validation and as the source for all subsequent transformations required to build the dimension and fact tables in the Gold layer.

In [0]:
# Define source silver and bronze table
SILVER_TABLE = "workspace.weather_data.weather_silver"
BRONZE_TABLE = "workspace.weather_data.weather_raw"
# Load transformed data from bronze and silver tables
silver_df = spark.read.table(SILVER_TABLE)
bronze_df = spark.read.table(BRONZE_TABLE)

# Display schemas and few records from both tables
print("Bronze table schema:")
bronze_df.printSchema()
print("Silver table schema:")
silver_df.printSchema()

print("\nBronze table records:")
display(bronze_df.limit(10))
print("\nSilver table records:")
display(silver_df.limit(10))

%md
# Validate Record Count Consistency Between Bronze and Silver Layers

This step compares the total number of records stored in the Bronze and Silver layers.

The Silver layer applies data transformations without adding or removing records.

The record counts must match between both layers.

The pipeline stops if a difference in record counts is detected.

In [0]:
# Counting rows in Bronze and Silver Tables
rows_bronze = bronze_df.count()
rows_silver = silver_df.count()
# Check if the number of rows in the silver table is equal to the number of rows in the bronze table
if rows_bronze != rows_silver:
   raise ValueError(f"Data quality check failed: Record count in bronze table: {rows_bronze}; Record count in silver table: {rows_silver}")
else:
    print(f"Data quality check passed: There are {rows_bronze} rows in both tables")

# Create the City Dimension Table

Starting from this step, SQL is used to perform data transformations and create dimension and fact tables in the Gold layer.

This approach separates data ingestion and preparation, handled with PySpark, from analytical data modeling, implemented using SQL.

The `dim_city` dimension table is created using data from the Silver layer.

The transformation consists of the following operations:

- Extract unique combinations of city names and geographic coordinates.
- Generate a unique surrogate key (`city_id`) for each city using the `ROW_NUMBER()` window function.
- Store the resulting dataset as a Delta table.

The `dim_city` table contains one record per city and serves as a reference dimension for the `fact_weather` table.

The table is created using `CREATE OR REPLACE TABLE`, allowing the transformation to be rerun and the dimension table to be fully refreshed.

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.weather_data.dim_city
  USING DELTA AS
SELECT
  -- Adding a surrogate key to the dim_city table
  ROW_NUMBER() OVER (ORDER BY city) AS city_id,
  *
FROM
  (
    -- Using subquery to select distinct city, latitude, longitude
    SELECT
    DISTINCT
      city,
      latitude,
      longitude
    FROM
      workspace.weather_data.weather_silver
  )

# Create the Date Dimension Table

This step creates the `dim_date` dimension table in the Gold layer using a dynamically generated calendar.

The calendar is generated independently of the individual records in the Silver layer. The Silver table is used only to determine the latest available date.

The transformation consists of the following operations:

- Define a fixed calendar start date of January 1, 2020.
- Retrieve the maximum available date from the Silver layer to determine the calendar end date.
- Generate a continuous sequence of calendar dates using the `SEQUENCE` and `EXPLODE` functions.
- Generate a surrogate key (`date_key`) for each date using the `yyyyMMdd` format.
- Derive calendar attributes, including day, month, month name, quarter, year, day of the week, and weekend indicator.
- Store the resulting dataset as a Delta table.

The `dim_date` table contains one record per calendar day, including dates without corresponding weather measurements.

The dynamic end date supports future incremental loading by allowing the calendar to expand as new dates become available in the Silver layer.

The initial table is created using `CREATE OR REPLACE TABLE`. Incremental loading will be implemented as a separate process in a subsequent development step.

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.weather_data.dim_date
  USING DELTA AS
WITH dates AS (
  SELECT
    EXPLODE(SEQUENCE(DATE "2020-01-01", MAX(DATE), INTERVAL 1 DAY)) AS full_date
  FROM
    workspace.weather_data.weather_silver
)
SELECT
  CAST(DATE_FORMAT(full_date, "yyyyMMdd") AS INT) AS date_key,
  full_date,
  DAY(full_date) AS day,
  MONTH(full_date) AS month,
  MONTHNAME(full_date) AS month_name,
  QUARTER(full_date) AS quarter,
  YEAR(full_date) AS year,
  WEEKDAY(full_date) + 1 AS day_of_week,
  CASE
    WHEN (weekday(full_date) + 1) IN (6, 7) THEN TRUE
    ELSE FALSE
  END AS is_weekend
FROM
  dates